# Multi-Agent OTel Evaluation Framework — Demo

**Enterprise-grade GenAI observability** aligned with OpenTelemetry GenAI Semantic Conventions,
evaluated on the [Mind2Web](https://osu-nlp-group.github.io/Mind2Web/) benchmark.

| Component | What it does |
|---|---|
| `src/config.py` | Provider-agnostic LLM factory (Azure, OpenAI, Ollama, Groq…) |
| `src/tracer.py` | OTel-compliant span tree + lightweight ExecutionTrace |
| `src/monitors.py` | CostTracker + HealthMonitor (rolling window) |
| `src/safety.py` | PII, injection, harmful-content validation |
| `src/tools.py` | Hybrid real/mock web-navigation tools |
| `src/agents.py` | Baseline ReAct agent + multi-agent supervisor pipeline |
| `src/evaluator.py` | HybridEvaluator (rule + LLM-as-judge) + ToolCorrectnessEval |
| `src/visualizer.py` | Eval dashboard, trace tree, telemetry charts |

---
**Before running:** copy `.env.example` → `.env` and fill in your API credentials.

## Step 1 — Install & Import

In [ ]:
# Uncomment on first run:
# !pip install -r requirements.txt

import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

from src import (
    Config,
    TracingManager, HierarchicalTracer,
    CostTracker, HealthMonitor,
    SafetyValidator,
    load_mind2web,
    Mind2WebTask,
    create_baseline_agent, run_agent,
    create_multi_agent, run_multi_agent,
    HybridEvaluator, ToolCorrectnessEval,
    plot_eval_dashboard, plot_trace_tree,
    plot_telemetry_dashboard, plot_dataset_overview,
)

print('✅ All imports successful')

## Step 2 — Configuration

In [ ]:
# All settings come from .env — no hardcoded credentials.
# Override here if needed:
# import os; os.environ['AGENT_MODEL'] = 'gpt-4o-mini'

Config.setup_dirs()

print('Provider:', 'Azure OpenAI' if Config.API_VERSION else 'OpenAI / compatible endpoint')
print('Agent model:', Config.AGENT_MODEL)
print('Judge model:', Config.JUDGE_MODEL)
print('Output dir:', Config.OUTPUT_DIR)
print('Pass threshold:', Config.EVAL_PASS_THRESHOLD)

## Step 3 — Verify LLM connectivity

In [ ]:
from langchain_core.messages import HumanMessage

agent_llm = Config.create_llm(role='agent')
judge_llm = Config.create_llm(role='judge')

resp = agent_llm.invoke([HumanMessage(content='Say hello in 5 words.')])
print('Agent LLM ✅:', resp.content.strip())

resp = judge_llm.invoke([HumanMessage(content='Rate this plan: "Search then click." Return only a float 0-1.')])
print('Judge LLM ✅:', resp.content.strip())

## Step 4 — Load Mind2Web Dataset

In [ ]:
# Streams from HuggingFace and caches locally — skips download on subsequent runs.
raw_tasks = load_mind2web(Config.DATA_DIR, target_tasks=Config.MIND2WEB_TARGET_TASKS)
print(f'Loaded {len(raw_tasks)} tasks')
print('Sample:', raw_tasks[0]['confirmed_task'][:100])

In [ ]:
# Dataset overview charts
fig = plot_dataset_overview(raw_tasks)
fig.savefig(Config.OUTPUT_DIR / 'dataset_overview.png', dpi=120, bbox_inches='tight')

## Step 5 — Initialize Framework Components

In [ ]:
tracing_manager = TracingManager()
hier_tracer     = HierarchicalTracer()
cost_tracker    = CostTracker()
health_monitor  = HealthMonitor(window_size=50)

agent, judge_llm = create_baseline_agent(Config)
evaluator        = HybridEvaluator(judge_llm,
                                    pass_threshold=Config.EVAL_PASS_THRESHOLD,
                                    rule_weight=Config.RULE_WEIGHT,
                                    llm_weight=Config.LLM_WEIGHT)

print('✅ All components initialized')

## Step 6 — Dry Run (1 task)
Verify the full pipeline before running the full quick test.

In [ ]:
task = Mind2WebTask.from_dict(raw_tasks[0], idx=0)

agent_output, trace = run_agent(task, agent, tracing_manager, Config)

print(f'Latency:    {trace.latency_ms:.0f} ms')
print(f'Tool calls: {len(trace.tool_calls)}')
print(f'Cost:       ${trace.total_cost:.5f}')
print(f'Errors:     {len(trace.errors)}')
print('\nOutput preview:')
print(agent_output[:400])

# Reset so dry run does not pollute quick test results
tracing_manager.reset()
health_monitor.reset()

## Step 7 — Quick Test (10 tasks)
Full observability: tracing, evaluation, tool correctness, safety, cost, health.

In [ ]:
import pandas as pd

N = Config.QUICK_TEST_N
results = []

for i in range(N):
    task = Mind2WebTask.from_dict(raw_tasks[i], idx=i)
    print(f'[{i+1}/{N}] {task.website}: {task.confirmed_task[:60]}…')

    agent_output, trace = run_agent(task, agent, tracing_manager, Config)
    safety_result = SafetyValidator.validate_all(agent_output, task.confirmed_task)
    eval_result   = evaluator.evaluate(agent_output, task, safety_result)
    tool_result   = ToolCorrectnessEval.evaluate(agent_output, task.action_reprs, trace.tool_calls)

    health_monitor.log(i, eval_result.passed, eval_result.total_score,
                       trace.latency_ms, len(trace.errors))
    cost_tracker.log(i, Config.AGENT_MODEL,
                     trace.agent_input_tokens, trace.agent_output_tokens, trace.total_cost)

    status = '✅' if eval_result.passed else '❌'
    print(f'  {status} score={eval_result.total_score:.2f} '
          f'tool_f1={tool_result.f1:.2f} '
          f'cost=${trace.total_cost:.4f} '
          f'latency={trace.latency_ms:.0f}ms')

    results.append({
        'task_id':      i,
        'website':      task.website,
        'task_score':   eval_result.total_score,
        'task_passed':  eval_result.passed,
        'rule_score':   eval_result.rule_score,
        'llm_score':    eval_result.llm_score,
        'tool_f1':      tool_result.f1,
        'tool_precision': tool_result.precision,
        'tool_recall':  tool_result.recall,
        'latency_ms':   trace.latency_ms,
        'agent_tokens': trace.agent_input_tokens + trace.agent_output_tokens,
        'judge_tokens': trace.judge_input_tokens + trace.judge_output_tokens,
        'total_cost':   trace.total_cost,
        'safety_passed': SafetyValidator.is_safe(safety_result),
    })

df = pd.DataFrame(results)
print('\nDone!')

## Step 8 — Summary Statistics

In [ ]:
print('=== Task Completion ===')
print(f'Pass Rate:     {df["task_passed"].mean():.1%}')
print(f'Avg Score:     {df["task_score"].mean():.3f}')
print(f'Avg Rule:      {df["rule_score"].mean():.3f}')
print(f'Avg LLM:       {df["llm_score"].mean():.3f}')

print('\n=== Tool Correctness ===')
print(f'Avg F1:        {df["tool_f1"].mean():.3f}')
print(f'Avg Precision: {df["tool_precision"].mean():.3f}')
print(f'Avg Recall:    {df["tool_recall"].mean():.3f}')

print('\n=== Cost & Performance ===')
print(f'Total Cost:    ${df["total_cost"].sum():.4f}')
print(f'Avg Latency:   {df["latency_ms"].mean():.0f} ms')
print(f'P95 Latency:   {df["latency_ms"].quantile(0.95):.0f} ms')

print('\n=== Agent Health ===')
health = health_monitor.get_status()
print(f'Status:        {health["status"]}')
print(f'Success Rate:  {health["success_rate"]:.1%}')

print('\n=== Safety ===')
print(f'Safety Pass:   {df["safety_passed"].mean():.1%}')

## Step 9 — Visualizations

In [ ]:
# Evaluation dashboard
fig = plot_eval_dashboard(df, save_path=Config.OUTPUT_DIR / 'eval_dashboard.png')

In [ ]:
# Telemetry dashboard
fig = plot_telemetry_dashboard(df, save_path=Config.OUTPUT_DIR / 'telemetry_dashboard.png')

In [ ]:
# OTel trace tree for the first task
if tracing_manager.traces:
    # Use hier_tracer for span-level view if populated, else skip
    first_trace_id = list(hier_tracer.traces.keys())[0] if hier_tracer.traces else None
    if first_trace_id:
        fig = plot_trace_tree(hier_tracer, first_trace_id,
                              save_path=Config.OUTPUT_DIR / 'trace_tree.png')
    else:
        print('No OTel spans captured in quick test (spans are generated by multi-agent run).')

## Step 10 — Multi-Agent Run (optional)
Supervisor → Planner → Navigator → Validator pipeline with full OTel span tracing.

In [ ]:
multi_agents = create_multi_agent(Config)

ma_results = []
hier_tracer.reset()
tracing_manager.reset()

for i in range(min(3, len(raw_tasks))):   # Run 3 tasks as demo
    task = Mind2WebTask.from_dict(raw_tasks[i], idx=i)
    print(f'[{i+1}] {task.website}: {task.confirmed_task[:60]}…')

    output, trace = run_multi_agent(task, multi_agents, hier_tracer, tracing_manager, Config)
    safety_result = SafetyValidator.validate_all(output, task.confirmed_task)
    eval_result   = evaluator.evaluate(output, task, safety_result)
    tool_result   = ToolCorrectnessEval.evaluate(output, task.action_reprs, trace.tool_calls)

    print(f'  Score: {eval_result.total_score:.2f} | Tool F1: {tool_result.f1:.2f} | '
          f'Cost: ${trace.total_cost:.4f}')
    ma_results.append({'task_id': i, 'task_score': eval_result.total_score,
                       'tool_f1': tool_result.f1, 'total_cost': trace.total_cost})

df_ma = pd.DataFrame(ma_results)
print('\nMulti-agent avg score:', df_ma['task_score'].mean())

In [ ]:
# OTel trace tree from multi-agent run
if hier_tracer.traces:
    first_id = list(hier_tracer.traces.keys())[0]
    fig = plot_trace_tree(hier_tracer, first_id,
                          save_path=Config.OUTPUT_DIR / 'multi_agent_trace_tree.png')

# Export OTLP traces
path = hier_tracer.save_all_traces(Config.TRACE_DIR)
print(f'OTLP traces saved → {path}')

## Step 11 — Save Results

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

df.to_csv(Config.OUTPUT_DIR / f'quick_test_results_{ts}.csv', index=False)
print(f'Results saved → {Config.OUTPUT_DIR}')

print('\nCost summary:')
print(cost_tracker.get_summary())